<a href="https://colab.research.google.com/github/psvprasad2003/IIMV_CAPSTONE_FILES/blob/IIMV_CAPSTONE_CODE_FILES/01_acquire_and_organize_data_v3_10workers_incremental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 Acquire and Organize Data

Run cell-by-cell. Paste each cell output back into chat. This notebook extracts ECFR archives, inventories RAW files, converts them to TXT, and classifies whether TXT files are usable for event analysis.

In [ ]:
# CELL 01 - Setup paths and debug helpers
from pathlib import Path
from datetime import datetime
import os, sys, hashlib, tarfile, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import pandas as pd

SOURCE_DIR = Path('/raid3/e296408/Download/ECFR')
WORKDIR = Path('/raid3/e296408/All_ECFRs_working')
EXTRACT_DIR = WORKDIR / 'extracted'
TXT_ROOT = WORKDIR / 'txt'
LOG_DIR = WORKDIR / 'logs'
PATCHED_CONVERTER = Path('/raid3/e296408/raw_2_dld_linux_safe.py')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
MIN_WORKERS = 1
MAX_WORKERS = 1
#max(MIN_WORKERS, min(1, os.cpu_count() or MIN_WORKERS))

ARCHIVE_MANIFEST = LOG_DIR / 'archive_extraction_manifest.csv'
RAW_INVENTORY_FILE = LOG_DIR / 'raw_inventory_latest.csv'
CONVERSION_MANIFEST = LOG_DIR / 'raw_to_txt_conversion_manifest.csv'
TXT_SCAN_REPORT = LOG_DIR / 'txt_content_scan_report.csv'
USABLE_LIST_FILE = LOG_DIR / 'usable_flight_txt_files.csv'
METADATA_ONLY_LIST_FILE = LOG_DIR / 'metadata_only_txt_files.csv'

for p in [WORKDIR, EXTRACT_DIR, TXT_ROOT, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def banner(title):
    print('=' * 100); print(title); print('=' * 100)

def dbg(name, value=None):
    print(f'[DEBUG] {name}')
    if value is not None:
        print(value)

banner('SETUP COMPLETE')
for k,v in {'RUN_ID':RUN_ID,'SOURCE_DIR':SOURCE_DIR,'WORKDIR':WORKDIR,'EXTRACT_DIR':EXTRACT_DIR,'TXT_ROOT':TXT_ROOT,'LOG_DIR':LOG_DIR,'PATCHED_CONVERTER':PATCHED_CONVERTER,'MAX_WORKERS':MAX_WORKERS}.items():
    print(f'{k:<22}: {v}')
print('SOURCE_DIR exists:', SOURCE_DIR.exists())
print('Converter exists :', PATCHED_CONVERTER.exists())

SETUP COMPLETE
RUN_ID                : 20260919_120105
SOURCE_DIR            : /raid3/e296408/Download/ECFR
WORKDIR               : /raid3/e296408/All_ECFRs_working
EXTRACT_DIR           : /raid3/e296408/All_ECFRs_working/extracted
TXT_ROOT              : /raid3/e296408/All_ECFRs_working/txt
LOG_DIR               : /raid3/e296408/All_ECFRs_working/logs
PATCHED_CONVERTER     : /raid3/e296408/raw_2_dld_linux_safe.py
MAX_WORKERS           : 1
SOURCE_DIR exists: True
Converter exists : True


In [ ]:
# CELL 02 - Discover ECFR archives
exclude_parts = {'.venv', 'site-packages', '__pycache__'}
archive_files = []
for pattern in ['*.tar.gz', '*.tgz', '*.tar']:
    for p in SOURCE_DIR.rglob(pattern):
        if any(part in exclude_parts for part in p.parts):
            continue
        archive_files.append(p)
archive_files = sorted(set(archive_files))
banner('ARCHIVE DISCOVERY')
print(f'Archives found: {len(archive_files):,}')
for p in archive_files[:20]:
    print(p)
if len(archive_files) > 20:
    print(f'... and {len(archive_files)-20:,} more')

ARCHIVE DISCOVERY
Archives found: 31,360
/raid3/e296408/Download/ECFR/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41.tar.gz
/raid3/e296408/Download/ECFR/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47.tar.gz
/raid3/e296408/Download/ECFR/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-11-02-49.tar.gz
/raid3/e296408/Download/ECFR/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-16-09-36-51.tar.gz
/raid3/e296408/Download/ECFR/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-16-09-43-15.tar.gz
/raid3/e296408/Download/ECFR/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2026-03-08-23-50-49.tar.gz
/raid3/e296408/Download/ECFR/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2025-11-25-09-18-51.tar.gz
/raid3/e296408/Download/ECFR/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2025-11-27-04-19-29.tar.gz
/raid3/e296408/Download/ECFR/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2025-12-05-11-39-59.tar.gz
/raid3/e296408/Download/ECFR/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2025-1

In [ ]:
# =============================================================================
# CELL 03 - Extract archives incrementally with concurrent workers
#
# Behavior:
# - Existing archives are skipped when .extract_complete exists.
# - Only new or previously incomplete archives are extracted.
# - TAR compression is detected from file content using mode="r:*".
# - Handles uncompressed TAR files incorrectly named .tar.gz.
# - Rejects unsafe paths, symbolic links, and hard links.
# - Removes partial extraction folders before retry.
# - Writes the completion marker only after successful validation.
# =============================================================================

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import hashlib
import json
import shutil
import tarfile
import traceback

import pandas as pd


# -----------------------------------------------------------------------------
# 03.01 Configuration
# -----------------------------------------------------------------------------

MIN_EXTRACTION_WORKERS = 10

EXTRACTION_WORKERS = max(
    MIN_EXTRACTION_WORKERS,
    min(
        MAX_WORKERS,
        os.cpu_count()
        or MIN_EXTRACTION_WORKERS,
    ),
)


# Accept these extracted input types.
#
# RAW is the normal input to the converter.
# TXT is accepted because some archives may already contain converted text.
# Other files, including converter configuration files, remain extracted but
# do not count as the primary ECFR payload check.
EXTRACTED_PAYLOAD_SUFFIXES = {
    ".raw",
    ".txt",
}


# -----------------------------------------------------------------------------
# 03.02 Helpers
# -----------------------------------------------------------------------------

def archive_base_name(archive_path):
    """
    Remove the recognized archive suffix without relying on Path.stem,
    which would leave '.tar' when processing '.tar.gz'.
    """

    name = Path(archive_path).name

    for suffix in (
        ".tar.gz",
        ".tgz",
        ".tar",
    ):
        if name.lower().endswith(suffix):
            return name[
                :-len(suffix)
            ]

    return Path(name).stem


def archive_folder_name(archive_path):
    """
    Preserve the existing folder-name convention so previously created
    .extract_complete markers remain valid.
    """

    return archive_base_name(
        archive_path
    )


def read_archive_magic(archive_path):
    """
    Read initial bytes for audit/classification.
    """

    archive_path = Path(
        archive_path
    )

    with archive_path.open(
        "rb"
    ) as stream:
        return stream.read(16)


def classify_tar_container(
    archive_path,
):
    """
    Describe the TAR container using actual content rather than extension.

    This classification is informational. tarfile.open(mode='r:*')
    performs the authoritative format detection.
    """

    archive_path = Path(
        archive_path
    )

    magic = read_archive_magic(
        archive_path
    )

    filename_claims_gzip = (
        archive_path.name
        .lower()
        .endswith(
            (
                ".tar.gz",
                ".tgz",
            )
        )
    )

    gzip_magic = (
        magic[:2]
        == b"\x1f\x8b"
    )

    if gzip_magic:
        return (
            "GZIP_TAR"
        )

    if tarfile.is_tarfile(
        archive_path
    ):

        if filename_claims_gzip:
            return (
                "UNCOMPRESSED_TAR_"
                "WRONG_GZIP_EXTENSION"
            )

        return (
            "UNCOMPRESSED_TAR"
        )

    return (
        "NOT_A_READABLE_TAR"
    )


def validate_tar_members(
    tar,
    destination,
):
    """
    Validate every TAR member before extraction.

    Rejects:
    - Absolute paths
    - Paths escaping the destination
    - Symbolic links
    - Hard links
    """

    destination = Path(
        destination
    ).resolve()

    members = tar.getmembers()

    for member in members:

        member_name = str(
            member.name
        )

        member_path = (
            destination
            / member_name
        ).resolve()

        try:
            member_path.relative_to(
                destination
            )

        except ValueError as exc:
            raise RuntimeError(
                "Unsafe TAR member path: "
                f"{member_name}"
            ) from exc

        if (
            member.issym()
            or member.islnk()
        ):
            raise RuntimeError(
                "Symbolic and hard links "
                "are not accepted: "
                f"{member_name}"
            )

    return members


def extracted_payload_files(
    extract_folder,
):
    """
    Return extracted RAW and TXT payload files, case-insensitively.
    """

    extract_folder = Path(
        extract_folder
    )

    return sorted(
        path
        for path in extract_folder.rglob(
            "*"
        )
        if (
            path.is_file()
            and path.suffix.lower()
            in EXTRACTED_PAYLOAD_SUFFIXES
        )
    )


def extracted_all_files(
    extract_folder,
):
    """
    Count all extracted regular files, excluding the completion marker.
    """

    extract_folder = Path(
        extract_folder
    )

    return sorted(
        path
        for path in extract_folder.rglob(
            "*"
        )
        if (
            path.is_file()
            and path.name
            != ".extract_complete"
        )
    )


def safe_extract_archive(
    archive_path,
    extract_folder,
):
    """
    Extract a TAR archive using content-based compression detection.
    """

    archive_path = Path(
        archive_path
    )

    extract_folder = Path(
        extract_folder
    )

    extract_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    # r:* auto-detects:
    # - uncompressed TAR
    # - gzip TAR
    # - bz2 TAR
    # - xz TAR
    #
    # It therefore processes valid uncompressed TAR content even when
    # the file is incorrectly named '.tar.gz'.
    with tarfile.open(
        archive_path,
        mode="r:*",
    ) as tar:

        members = validate_tar_members(
            tar=tar,
            destination=extract_folder,
        )

        tar.extractall(
            path=extract_folder,
            members=members,
        )

    return members


# -----------------------------------------------------------------------------
# 03.03 Incremental archive worker
# -----------------------------------------------------------------------------

def extract_one_archive(
    item,
):

    idx, archive = item

    archive = Path(
        archive
    )

    folder_name = archive_folder_name(
        archive
    )

    extract_folder = (
        EXTRACT_DIR
        / folder_name
    )

    marker = (
        extract_folder
        / ".extract_complete"
    )

    base_result = {
        "archive_index":
            idx,

        "archive":
            str(archive),

        "archive_name":
            archive.name,

        "archive_size_bytes":
            (
                archive.stat().st_size
                if archive.is_file()
                else None
            ),

        "extract_folder":
            str(extract_folder),

        "archive_type":
            None,

        "member_count":
            0,

        "extracted_file_count":
            0,

        "raw_file_count":
            0,

        "txt_file_count":
            0,

        "status":
            None,

        "error_category":
            "",

        "error":
            "",
    }


    if marker.is_file():

        base_result[
            "status"
        ] = (
            "SKIPPED_ALREADY_EXTRACTED"
        )

        try:
            marker_payload = json.loads(
                marker.read_text(
                    encoding="utf-8"
                )
            )

            base_result[
                "archive_type"
            ] = marker_payload.get(
                "archive_type"
            )

            base_result[
                "member_count"
            ] = marker_payload.get(
                "member_count",
                0,
            )

            base_result[
                "extracted_file_count"
            ] = marker_payload.get(
                "extracted_file_count",
                0,
            )

            base_result[
                "raw_file_count"
            ] = marker_payload.get(
                "raw_file_count",
                0,
            )

            base_result[
                "txt_file_count"
            ] = marker_payload.get(
                "txt_file_count",
                0,
            )

        except Exception:
            # Older markers may contain plain text rather than JSON.
            pass

        return base_result


    try:

        if not archive.is_file():

            raise FileNotFoundError(
                f"Archive does not exist: "
                f"{archive}"
            )


        archive_type = classify_tar_container(
            archive
        )

        base_result[
            "archive_type"
        ] = archive_type


        if archive_type == (
            "NOT_A_READABLE_TAR"
        ):

            raise tarfile.ReadError(
                "File is not a readable TAR "
                "archive."
            )


        # Remove partial extraction output left by an earlier failure.
        #
        # A valid marker would already have returned above, so deleting this
        # folder does not remove a completed extraction.
        if extract_folder.exists():

            shutil.rmtree(
                extract_folder
            )


        extract_folder.mkdir(
            parents=True,
            exist_ok=True,
        )


        members = safe_extract_archive(
            archive_path=archive,
            extract_folder=extract_folder,
        )


        all_files = extracted_all_files(
            extract_folder
        )

        payload_files = (
            extracted_payload_files(
                extract_folder
            )
        )

        raw_files = [
            path
            for path in payload_files
            if path.suffix.lower()
            == ".raw"
        ]

        txt_files = [
            path
            for path in payload_files
            if path.suffix.lower()
            == ".txt"
        ]


        base_result[
            "member_count"
        ] = len(
            members
        )

        base_result[
            "extracted_file_count"
        ] = len(
            all_files
        )

        base_result[
            "raw_file_count"
        ] = len(
            raw_files
        )

        base_result[
            "txt_file_count"
        ] = len(
            txt_files
        )


        if not members:

            raise RuntimeError(
                "Archive contains no TAR members."
            )


        if not all_files:

            raise RuntimeError(
                "Archive extraction produced "
                "no regular files."
            )


        if (
            not raw_files
            and not txt_files
        ):

            # Do not silently mark the archive complete if no usable ECFR
            # payload was extracted. Configuration/support files may exist,
            # but the archive cannot yet feed RAW-to-TXT or TXT processing.
            raise RuntimeError(
                "Archive extraction produced no "
                "RAW or TXT payload files."
            )


        if archive_type == (
            "UNCOMPRESSED_TAR_"
            "WRONG_GZIP_EXTENSION"
        ):

            status = (
                "SUCCESS_UNCOMPRESSED_TAR_"
                "WRONG_EXTENSION"
            )

        elif archive_type == (
            "GZIP_TAR"
        ):

            status = (
                "SUCCESS_GZIP_TAR"
            )

        else:

            status = (
                "SUCCESS_UNCOMPRESSED_TAR"
            )


        marker_payload = {
            "status":
                "COMPLETE",

            "archive":
                str(archive),

            "archive_type":
                archive_type,

            "member_count":
                len(members),

            "extracted_file_count":
                len(all_files),

            "raw_file_count":
                len(raw_files),

            "txt_file_count":
                len(txt_files),
        }


        # Write the marker only after extraction and payload validation pass.
        marker.write_text(
            json.dumps(
                marker_payload,
                indent=2,
            ),
            encoding="utf-8",
        )


        base_result[
            "status"
        ] = status

        return base_result


    except tarfile.ReadError as exc:

        base_result[
            "status"
        ] = (
            "FAILED_NOT_READABLE_TAR"
        )

        base_result[
            "error_category"
        ] = (
            "TAR_READ_ERROR"
        )

        base_result[
            "error"
        ] = repr(
            exc
        )


    except PermissionError as exc:

        base_result[
            "status"
        ] = (
            "FAILED_PERMISSION"
        )

        base_result[
            "error_category"
        ] = (
            "PERMISSION_ERROR"
        )

        base_result[
            "error"
        ] = repr(
            exc
        )


    except FileNotFoundError as exc:

        base_result[
            "status"
        ] = (
            "FAILED_ARCHIVE_MISSING"
        )

        base_result[
            "error_category"
        ] = (
            "FILE_NOT_FOUND"
        )

        base_result[
            "error"
        ] = repr(
            exc
        )


    except Exception as exc:

        error_text = str(
            exc
        )

        if (
            "Unsafe TAR member path"
            in error_text
            or "links are not accepted"
            in error_text
        ):

            status = (
                "FAILED_UNSAFE_ARCHIVE_MEMBER"
            )

            category = (
                "UNSAFE_ARCHIVE_MEMBER"
            )

        elif (
            "no RAW or TXT payload"
            in error_text
        ):

            status = (
                "FAILED_NO_ECFR_PAYLOAD"
            )

            category = (
                "NO_RAW_OR_TXT"
            )

        elif (
            "no regular files"
            in error_text
            or "no TAR members"
            in error_text
        ):

            status = (
                "FAILED_EMPTY_ARCHIVE"
            )

            category = (
                "EMPTY_ARCHIVE"
            )

        else:

            status = (
                "FAILED_OTHER"
            )

            category = (
                type(exc).__name__
            )


        base_result[
            "status"
        ] = status

        base_result[
            "error_category"
        ] = category

        base_result[
            "error"
        ] = (
            traceback.format_exc(
                limit=6
            )
        )


    # Remove incomplete output so the next incremental run gets a clean retry.
    try:

        if (
            extract_folder.exists()
            and not marker.exists()
        ):

            shutil.rmtree(
                extract_folder
            )

    except Exception as cleanup_exc:

        base_result[
            "error"
        ] = (
            str(
                base_result[
                    "error"
                ]
            )
            + "\n"
            + "PARTIAL_FOLDER_CLEANUP_FAILED: "
            + repr(
                cleanup_exc
            )
        )


    return base_result


# -----------------------------------------------------------------------------
# 03.04 Build incremental work list
# -----------------------------------------------------------------------------

rows = []

items = []


for idx, archive in enumerate(
    archive_files,
    start=1,
):

    archive = Path(
        archive
    )

    folder_name = (
        archive_folder_name(
            archive
        )
    )

    extract_folder = (
        EXTRACT_DIR
        / folder_name
    )

    marker = (
        extract_folder
        / ".extract_complete"
    )


    if marker.is_file():

        rows.append(
            {
                "archive_index":
                    idx,

                "archive":
                    str(archive),

                "archive_name":
                    archive.name,

                "archive_size_bytes":
                    (
                        archive.stat().st_size
                        if archive.is_file()
                        else None
                    ),

                "extract_folder":
                    str(extract_folder),

                "archive_type":
                    None,

                "member_count":
                    0,

                "extracted_file_count":
                    0,

                "raw_file_count":
                    0,

                "txt_file_count":
                    0,

                "status":
                    "SKIPPED_ALREADY_EXTRACTED",

                "error_category":
                    "",

                "error":
                    "",
            }
        )

    else:

        items.append(
            (
                idx,
                archive,
            )
        )


print(
    f"Archives total      : "
    f"{len(archive_files):,}"
)

print(
    f"Already extracted  : "
    f"{len(rows):,}"
)

print(
    f"New/uncompleted    : "
    f"{len(items):,}"
)

print(
    f"Concurrent workers : "
    f"{EXTRACTION_WORKERS:,}"
)


# -----------------------------------------------------------------------------
# 03.05 Concurrent extraction
# -----------------------------------------------------------------------------

if items:

    with ThreadPoolExecutor(
        max_workers=EXTRACTION_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                extract_one_archive,
                item,
            ):
            item

            for item in items
        }


        completed = 0


        for future in as_completed(
            futures
        ):

            item = futures[
                future
            ]

            try:

                result = future.result()

            except Exception as exc:

                idx, archive = item

                result = {
                    "archive_index":
                        idx,

                    "archive":
                        str(archive),

                    "archive_name":
                        Path(
                            archive
                        ).name,

                    "archive_size_bytes":
                        (
                            Path(
                                archive
                            ).stat().st_size
                            if Path(
                                archive
                            ).is_file()
                            else None
                        ),

                    "extract_folder":
                        "",

                    "archive_type":
                        None,

                    "member_count":
                        0,

                    "extracted_file_count":
                        0,

                    "raw_file_count":
                        0,

                    "txt_file_count":
                        0,

                    "status":
                        "FAILED_WORKER_EXCEPTION",

                    "error_category":
                        type(
                            exc
                        ).__name__,

                    "error":
                        traceback.format_exc(
                            limit=6
                        ),
                }


            rows.append(
                result
            )

            completed += 1


            if str(
                result[
                    "status"
                ]
            ).startswith(
                "FAILED"
            ):

                print(
                    "FAILED:",
                    result[
                        "archive"
                    ],
                )

                print(
                    result[
                        "status"
                    ],
                    result[
                        "error_category"
                    ],
                )

                print(
                    str(
                        result[
                            "error"
                        ]
                    )[:1000]
                )


            elif (
                completed == 1
                or completed % 100 == 0
                or completed == len(items)
            ):

                print(
                    f"[{completed:,}/"
                    f"{len(items):,}] "
                    f"{result['status']}: "
                    f"{Path(result['archive']).name}"
                )


# -----------------------------------------------------------------------------
# 03.06 Publish manifest and summary
# -----------------------------------------------------------------------------

archive_manifest_df = pd.DataFrame(
    rows
)


if not archive_manifest_df.empty:

    archive_manifest_df = (
        archive_manifest_df
        .sort_values(
            [
                "archive_index",
                "archive",
            ]
        )
        .reset_index(
            drop=True
        )
    )


archive_manifest_df.to_csv(
    ARCHIVE_MANIFEST,
    index=False,
)


successful_statuses = {
    "SUCCESS_GZIP_TAR",
    "SUCCESS_UNCOMPRESSED_TAR",
    "SUCCESS_UNCOMPRESSED_TAR_WRONG_EXTENSION",
}


success = int(
    archive_manifest_df[
        "status"
    ].isin(
        successful_statuses
    ).sum()
)


skipped = int(
    archive_manifest_df[
        "status"
    ].eq(
        "SKIPPED_ALREADY_EXTRACTED"
    ).sum()
)


failed = int(
    archive_manifest_df[
        "status"
    ]
    .astype(str)
    .str.startswith(
        "FAILED"
    )
    .sum()
)


wrong_extension_success = int(
    archive_manifest_df[
        "status"
    ].eq(
        "SUCCESS_UNCOMPRESSED_TAR_WRONG_EXTENSION"
    ).sum()
)


banner(
    "EXTRACTION SUMMARY"
)


print(
    f"Success                  : "
    f"{success:,}"
)

print(
    f"  Wrong-extension TARs   : "
    f"{wrong_extension_success:,}"
)

print(
    f"Skipped                  : "
    f"{skipped:,}"
)

print(
    f"Failed                   : "
    f"{failed:,}"
)

print(
    "Manifest:",
    ARCHIVE_MANIFEST,
)


print(
    "\nSTATUS DISTRIBUTION"
)


print(
    archive_manifest_df[
        "status"
    ].value_counts(
        dropna=False
    )
)


if failed:

    failure_summary = (
        archive_manifest_df.loc[
            archive_manifest_df[
                "status"
            ]
            .astype(str)
            .str.startswith(
                "FAILED"
            )
        ]
        .groupby(
            [
                "status",
                "error_category",
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="archives"
        )
        .sort_values(
            [
                "archives",
                "status",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )


    failure_summary_path = (
        LOG_DIR
        / (
            "archive_extraction_"
            "failure_summary_"
            f"{RUN_ID}.csv"
        )
    )


    failure_summary.to_csv(
        failure_summary_path,
        index=False,
    )


    print(
        "\nFailure summary:",
        failure_summary_path,
    )


    display(
        failure_summary
    )

Archives total      : 31,360
Already extracted  : 31,321
New/uncompleted    : 39
Concurrent workers : 10
FAILED: /raid3/e296408/Download/ECFR/ECFR_16b3d99d-eca4-43a3-8b9f-b6236383abdc_2026-01-11-01-15-58.tar.gz
FAILED_NOT_READABLE_TAR TAR_READ_ERROR
ReadError('File is not a readable TAR archive.')
FAILED: /raid3/e296408/Download/ECFR/ECFR_2a67a2b5-98e5-491d-b80d-09838ecc9676_2025-08-18-00-05-57.tar.gz
FAILED_NOT_READABLE_TAR TAR_READ_ERROR
ReadError('File is not a readable TAR archive.')
FAILED: /raid3/e296408/Download/ECFR/ECFR_16b3d99d-eca4-43a3-8b9f-b6236383abdc_2026-01-19-13-35-37.tar.gz
FAILED_NOT_READABLE_TAR TAR_READ_ERROR
ReadError('File is not a readable TAR archive.')
FAILED: /raid3/e296408/Download/ECFR/ECFR_75407804-1102-48f0-865c-9b291bc4b627_2025-11-25-13-48-06.tar.gz
FAILED_NOT_READABLE_TAR TAR_READ_ERROR
ReadError('File is not a readable TAR archive.')
FAILED: /raid3/e296408/Download/ECFR/ECFR_88070c99-7463-4d24-965e-4b5d0c75f45f_2025-12-05-06-33-00.tar.gz
FAILED_NOT_RE

,status,error_category,archives
1,FAILED_NOT_READABLE_TAR,TAR_READ_ERROR,33
3,FAILED_PERMISSION,PERMISSION_ERROR,3
0,FAILED_EMPTY_ARCHIVE,EMPTY_ARCHIVE,2
2,FAILED_OTHER,EOFError,1


In [ ]:
from pathlib import Path
import tarfile
import zipfile


archive = Path(
    "/raid3/e296408/Download/ECFR/"
    "ECFR_9HVCC_2024-11-26-12-40-26.tar.gz"
)


with archive.open("rb") as stream:
    magic = stream.read(16)


print(
    {
        "exists":
            archive.is_file(),

        "size_bytes":
            (
                archive.stat().st_size
                if archive.is_file()
                else None
            ),

        "first_16_bytes_hex":
            magic.hex(),

        "gzip_magic":
            magic[:2] == b"\x1f\x8b",

        "zip_magic":
            magic[:4] in {
                b"PK\x03\x04",
                b"PK\x05\x06",
                b"PK\x07\x08",
            },

        "tar_readable_auto":
            tarfile.is_tarfile(
                archive
            ),

        "zip_readable":
            zipfile.is_zipfile(
                archive
            ),
    }
)

{'exists': True, 'size_bytes': 1146368, 'first_16_bytes_hex': '454346525f39482d5643435f32303234', 'gzip_magic': False, 'zip_magic': False, 'tar_readable_auto': True, 'zip_readable': False}


In [ ]:
# CELL 04 - Build RAW file inventory with stable raw_key

def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

raw_files = sorted(list(EXTRACT_DIR.rglob('*.raw')) + list(EXTRACT_DIR.rglob('*.RAW')))
rows=[]
print(f'RAW files found: {len(raw_files):,}')
for idx, raw_file in enumerate(raw_files, start=1):
    if idx == 1 or idx % 1000 == 0:
        print(f'Hashing {idx:,}/{len(raw_files):,}: {raw_file.name}')
    stat = raw_file.stat()
    rel = raw_file.relative_to(EXTRACT_DIR)
    sha = file_sha256(raw_file)
    rows.append({'raw_key':f'{rel}|{stat.st_size}|{sha}','raw_file':str(raw_file),'relative_raw_path':str(rel),'raw_size_bytes':stat.st_size,'raw_sha256':sha,'raw_mtime_reference_only':datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds')})
raw_inventory_df = pd.DataFrame(rows)
raw_inventory_df.to_csv(RAW_INVENTORY_FILE, index=False)
banner('RAW INVENTORY SUMMARY')
print('Rows       :', len(raw_inventory_df))
print('Unique keys:', raw_inventory_df['raw_key'].nunique() if len(raw_inventory_df) else 0)
print('Output     :', RAW_INVENTORY_FILE)
display(raw_inventory_df.head())

RAW files found: 125,325
Hashing 1/125,325: ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_1.raw
Hashing 1,000/125,325: ECFR_01a4a34c-9b42-4f27-841a-797ad42ac8cf_2026-04-15-08-38-50_1_444444_2.raw
Hashing 2,000/125,325: ECFR_0e2cdd21-3c2b-4391-8d6f-1935bc7fe177_2025-12-08-02-10-22_1_136553_2.raw
Hashing 3,000/125,325: ECFR_14cf806f-49c9-42bc-9f63-351fad5a4e7d_2026-04-07-19-57-27_2_145294_2.raw
Hashing 4,000/125,325: ECFR_16b3d99d-eca4-43a3-8b9f-b6236383abdc_2026-05-07-17-41-34_1_P131499_2.raw
Hashing 5,000/125,325: ECFR_1c70cd55-71c7-4f3e-8bfa-d02f95d2f125_2000-00-00-00-07-25_2_131204_2.raw
Hashing 6,000/125,325: ECFR_20db5ee8-02ae-43cf-bf28-d97985fb1cdc_2026-07-26-10-05-51_1_136505_1.raw
Hashing 7,000/125,325: ECFR_242c978f-2566-4e81-afe0-07e4809b3a97_2025-12-24-06-26-00_2_136506_1.raw
Hashing 8,000/125,325: ECFR_2e619411-91a0-4a40-b3c4-432d75ab8955_2026-02-13-20-13-58_2_131490_1.raw
Hashing 9,000/125,325: ECFR_2f8a166a-0d55-48f7-882e-0fbbd6c135b0_2026-09-17-22

,raw_key,raw_file,relative_raw_path,raw_size_bytes,raw_sha256,raw_mtime_reference_only
0,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_1.raw|40496|d93ee8fd54cdffa8e082190a64edb57170edcfc2d3ac6ebb05dc96a1fc49f063,/raid3/e296408/All_ECFRs_working/extracted/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_1.raw,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_1.raw,40496,d93ee8fd54cdffa8e082190a64edb57170edcfc2d3ac6ebb05dc96a1fc49f063,2025-12-06T06:26:47
1,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_2.raw|40496|6f7d4e102ce877329320bf3cc70c73e7e5fafb09a86101cef3bd315c27cf253c,/raid3/e296408/All_ECFRs_working/extracted/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_2.raw,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_1_136505_2.raw,40496,6f7d4e102ce877329320bf3cc70c73e7e5fafb09a86101cef3bd315c27cf253c,2025-12-06T06:26:47
2,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_2_136506_1.raw|40496|ca9faaecaf08b722781a56c5eae7291797a855c3af1a49137238093a25807111,/raid3/e296408/All_ECFRs_working/extracted/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_2_136506_1.raw,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_2_136506_1.raw,40496,ca9faaecaf08b722781a56c5eae7291797a855c3af1a49137238093a25807111,2025-12-06T06:26:47
3,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_2_136506_2.raw|40496|5393b4d73b0823c29fa23c910b9dbd2fe2f28748787a9a35dc9ef310cad6e2d5,/raid3/e296408/All_ECFRs_working/extracted/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_2_136506_2.raw,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-06-06-28-41_2_136506_2.raw,40496,5393b4d73b0823c29fa23c910b9dbd2fe2f28748787a9a35dc9ef310cad6e2d5,2025-12-06T06:26:47
4,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47_1_136505_1.raw|40496|d93ee8fd54cdffa8e082190a64edb57170edcfc2d3ac6ebb05dc96a1fc49f063,/raid3/e296408/All_ECFRs_working/extracted/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47_1_136505_1.raw,ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47/ECFRdata/ECFR_0079778c-15cd-4127-b0f2-80d6ca604072_2025-12-15-10-57-47_1_136505_1.raw,40496,d93ee8fd54cdffa8e082190a64edb57170edcfc2d3ac6ebb05dc96a1fc49f063,2025-12-15T10:56:13


In [ ]:
# CELL 05 - Initialize conversion manifest
conversion_columns = ['run_id','timestamp','raw_key','raw_file','relative_raw_path','raw_size_bytes','raw_sha256','txt_file','dld_file','status','return_code','stdout','stderr']
if not CONVERSION_MANIFEST.exists():
    pd.DataFrame(columns=conversion_columns).to_csv(CONVERSION_MANIFEST, index=False)
    print('Created:', CONVERSION_MANIFEST)
else:
    conversion_df = pd.read_csv(CONVERSION_MANIFEST)
    print('Existing manifest:', CONVERSION_MANIFEST)
    print('Rows:', len(conversion_df))
    print('Columns:', conversion_df.columns.tolist())

Existing manifest: /raid3/e296408/All_ECFRs_working/logs/raw_to_txt_conversion_manifest.csv
Rows: 145702
Columns: ['run_id', 'timestamp', 'raw_key', 'raw_file', 'relative_raw_path', 'raw_size_bytes', 'raw_sha256', 'txt_file', 'dld_file', 'status', 'return_code', 'stdout', 'stderr']


In [ ]:
# CELL 06 - Identify pending RAW conversions
raw_inventory_df = pd.read_csv(RAW_INVENTORY_FILE)
conversion_df = pd.read_csv(CONVERSION_MANIFEST)
success_statuses = {'SUCCESS','SKIPPED_TXT_EXISTS','ADOPTED_EXISTING_TXT'}
successful_keys = set(conversion_df.loc[conversion_df['status'].isin(success_statuses), 'raw_key'].dropna().astype(str)) if len(conversion_df) else set()
pending=[]; adopted=[]
for _, row in raw_inventory_df.iterrows():
    raw_key = str(row['raw_key'])
    if raw_key in successful_keys:
        continue
    rel = Path(row['relative_raw_path'])
    txt_file = (TXT_ROOT / rel).with_suffix('.txt')
    dld_file = (TXT_ROOT / rel).with_suffix('.dld')
    if txt_file.exists() and txt_file.stat().st_size > 0:
        adopted.append({'run_id':RUN_ID,'timestamp':datetime.now().isoformat(timespec='seconds'),'raw_key':raw_key,'raw_file':row['raw_file'],'relative_raw_path':str(rel),'raw_size_bytes':row['raw_size_bytes'],'raw_sha256':row['raw_sha256'],'txt_file':str(txt_file),'dld_file':str(dld_file),'status':'ADOPTED_EXISTING_TXT','return_code':'','stdout':'','stderr':''})
    else:
        pending.append({'raw_key':raw_key,'raw_file':row['raw_file'],'relative_raw_path':str(rel),'raw_size_bytes':row['raw_size_bytes'],'raw_sha256':row['raw_sha256'],'txt_file':str(txt_file),'dld_file':str(dld_file)})
if adopted:
    pd.DataFrame(adopted).to_csv(CONVERSION_MANIFEST, mode='a', index=False, header=False)
pending_raw_df = pd.DataFrame(pending)
banner('PENDING CONVERSION SUMMARY')
print('Total RAW files        :', len(raw_inventory_df))
print('Successful raw keys    :', len(successful_keys))
print('Adopted existing TXT   :', len(adopted))
print('Pending conversions    :', len(pending_raw_df))
display(pending_raw_df.head())

PENDING CONVERSION SUMMARY
Total RAW files        : 125325
Successful raw keys    : 111288
Adopted existing TXT   : 0
Pending conversions    : 14060


,raw_key,raw_file,relative_raw_path,raw_size_bytes,raw_sha256,txt_file,dld_file
0,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_1_118327_2.raw|244428|268021a11965372da9a43b1503b8e6104dba9802f62afb367e2596f0c8d09291,/raid3/e296408/All_ECFRs_working/extracted/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_1_118327_2.raw,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_1_118327_2.raw,244428,268021a11965372da9a43b1503b8e6104dba9802f62afb367e2596f0c8d09291,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_1_118327_2.txt,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_1_118327_2.dld
1,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_1.raw|244768|96fe91fdb30f3b3dffaf34871bee87a0fdc248bb1a752eeda30347ceb99508ea,/raid3/e296408/All_ECFRs_working/extracted/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_1.raw,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_1.raw,244768,96fe91fdb30f3b3dffaf34871bee87a0fdc248bb1a752eeda30347ceb99508ea,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_1.txt,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_1.dld
2,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_2.raw|245056|786ecf3d1bd7868f55cb7a9f80cfd2d166bf13f21c033c08a583979c75695e07,/raid3/e296408/All_ECFRs_working/extracted/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_2.raw,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_2.raw,245056,786ecf3d1bd7868f55cb7a9f80cfd2d166bf13f21c033c08a583979c75695e07,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_2.txt,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_2_118328_2.dld
3,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24_1_118327_1.raw|166500|401e4dc408120b8fd9544e9622a554bf32e133e30fecccc8a442675136529822,/raid3/e296408/All_ECFRs_working/extracted/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24_1_118327_1.raw,ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24_1_118327_1.raw,166500,401e4dc408120b8fd9544e9622a554bf32e133e30fecccc8a442675136529822,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24/ECFRdata/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-12-24_1_118327_1.txt,/raid3/e296408/All_ECFRs_working/txt/ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-09-11-16-

In [ ]:
# CELL 07 - Convert pending RAW files using at least 10 concurrent workers
# Only pending_raw_df is converted. Successful manifest keys and existing TXT
# outputs were removed/adopted in Cell 06.
MAX_CONVERSIONS = None  # Example: 25 for smoke test
KEEP_DLD = False
if not PATCHED_CONVERTER.exists():
    raise FileNotFoundError(f'Converter not found: {PATCHED_CONVERTER}')
selected = pending_raw_df.head(MAX_CONVERSIONS).copy() if MAX_CONVERSIONS else pending_raw_df.copy()
print(f'Selected new/pending conversions: {len(selected):,}')
print(f'Concurrent workers              : {MAX_WORKERS:,}')
env=os.environ.copy(); env['PYTHONUTF8']='1'; env['PYTHONIOENCODING']='utf-8'

def convert_one_pending(row_dict):
    row=pd.Series(row_dict)
    raw_file=Path(row['raw_file']); txt_file=Path(row['txt_file']); dld_file=Path(row['dld_file'])
    txt_file.parent.mkdir(parents=True, exist_ok=True)
    # Recheck at execution time so a concurrent/restarted run never reconverts
    # a RAW whose TXT was created after Cell 06 built pending_raw_df.
    if txt_file.exists() and txt_file.stat().st_size > 0:
        return {'run_id':RUN_ID,'timestamp':datetime.now().isoformat(timespec='seconds'),'raw_key':row['raw_key'],'raw_file':str(raw_file),'relative_raw_path':row['relative_raw_path'],'raw_size_bytes':row['raw_size_bytes'],'raw_sha256':row['raw_sha256'],'txt_file':str(txt_file),'dld_file':str(dld_file),'status':'SKIPPED_TXT_EXISTS','return_code':'','stdout':'','stderr':''}
    cmd=[sys.executable,str(PATCHED_CONVERTER),'-i',str(raw_file),'-o',str(dld_file),'-t',str(txt_file)]
    try:
        result=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,encoding='utf-8',errors='replace',cwd=PATCHED_CONVERTER.parent,env=env)
        stdout_text=result.stdout or ''; stderr_text=result.stderr or ''
        if txt_file.exists() and txt_file.stat().st_size > 0:
            status='SUCCESS'
            if not KEEP_DLD and dld_file.exists(): dld_file.unlink()
        else:
            status='FAILED_CONFIG_MISSING' if 'Corrupted or Missing ECFR config' in stdout_text else 'FAILED_OTHER'
        return {'run_id':RUN_ID,'timestamp':datetime.now().isoformat(timespec='seconds'),'raw_key':row['raw_key'],'raw_file':str(raw_file),'relative_raw_path':row['relative_raw_path'],'raw_size_bytes':row['raw_size_bytes'],'raw_sha256':row['raw_sha256'],'txt_file':str(txt_file),'dld_file':str(dld_file),'status':status,'return_code':result.returncode,'stdout':stdout_text.strip(),'stderr':stderr_text.strip()}
    except Exception as ex:
        return {'run_id':RUN_ID,'timestamp':datetime.now().isoformat(timespec='seconds'),'raw_key':row['raw_key'],'raw_file':str(raw_file),'relative_raw_path':row['relative_raw_path'],'raw_size_bytes':row['raw_size_bytes'],'raw_sha256':row['raw_sha256'],'txt_file':str(txt_file),'dld_file':str(dld_file),'status':'EXCEPTION','return_code':'','stdout':'','stderr':str(ex)}

run_rows=[]
records=selected.to_dict('records')
if records:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures={executor.submit(convert_one_pending,row):row['raw_key'] for row in records}
        for idx,future in enumerate(as_completed(futures),start=1):
            result=future.result(); run_rows.append(result)
            if idx==1 or idx%100==0 or idx==len(records):
                print(f'[{idx:,}/{len(records):,}] {result["status"]}: {Path(result["raw_file"]).name}')
run_df=pd.DataFrame(run_rows)
if len(run_df):
    run_df.to_csv(CONVERSION_MANIFEST,mode='a',index=False,header=False)
    report=LOG_DIR/f'raw_to_txt_conversion_report_{RUN_ID}.csv'
    run_df.to_csv(report,index=False)
else:
    report=None
success=int(run_df['status'].isin(['SUCCESS','SKIPPED_TXT_EXISTS']).sum()) if len(run_df) else 0
failed=int((~run_df['status'].isin(['SUCCESS','SKIPPED_TXT_EXISTS'])).sum()) if len(run_df) else 0
banner('RAW TO TXT CONVERSION SUMMARY')
print('Success/skipped:',success)
print('Failed         :',failed)
print('Report         :',report)
if len(run_df): display(run_df['status'].value_counts(dropna=False))


Selected new/pending conversions: 14,060
Concurrent workers              : 1
[1/14,060] SUCCESS: ECFR_00d2c773-63ca-484d-9bf4-7c3890b2f5a5_2026-08-26-20-06-11_1_118327_2.raw
[100/14,060] SUCCESS: ECFR_0138974e-12d8-4389-b0a9-057d748ee32d_2026-09-03-10-37-45_2_P136292_1.raw
[200/14,060] FAILED_OTHER: ECFR_0e2cdd21-3c2b-4391-8d6f-1935bc7fe177_2024-12-28-05-47-43_2_136506_1.raw
[300/14,060] FAILED_OTHER: ECFR_106647f0-5d07-4278-b233-7a8856a31dea_2025-12-04-08-45-34_1_136505_2.raw
[400/14,060] FAILED_OTHER: ECFR_16b3d99d-eca4-43a3-8b9f-b6236383abdc_2026-04-10-15-59-09_1_P131499_1.raw
[500/14,060] SUCCESS: ECFR_16b3d99d-eca4-43a3-8b9f-b6236383abdc_2026-09-09-22-15-39_2_P131555_2.raw
[600/14,060] FAILED_OTHER: ECFR_1c70cd55-71c7-4f3e-8bfa-d02f95d2f125_2026-04-24-02-27-53_2_P131403_2.raw
[700/14,060] FAILED_OTHER: ECFR_1c70cd55-71c7-4f3e-8bfa-d02f95d2f125_2000-00-00-00-24-52_2_131204_2.raw
[800/14,060] SUCCESS: ECFR_1c70cd55-71c7-4f3e-8bfa-d02f95d2f125_2026-09-08-21-44-49_1_P131197_1.raw
[900

status
SUCCESS                  6946
FAILED_OTHER             6846
FAILED_CONFIG_MISSING     268
Name: count, dtype: int64

In [ ]:
# CELL 08 - Scan TXT content and classify usable ECFR files - V2
# IMPORTANT:
#   Converted TXT files do NOT have EVENT_ID yet. EVENT_ID is created later by the merge step.
#   Therefore, classification must look for actual ECFR tabular sections and data rows,
#   not EVENT_ID.

from pathlib import Path
from io import StringIO
import pandas as pd

TXT_SECTION_NAMES = {
    'STARTS', 'TKO_PERFORMANCE', 'CLM_PERFORMANCE', 'CRU_PERFORMANCE', 'ROLLDOWN',
    'INCIDENT_RECORD_A', 'INCIDENT_RECORD_B', 'ENGINE_REDLINE', 'IGNITION_RECORD', 'CHIP',
    'FF_ATTRIBUTES', 'ENGINE_TIME_AT_PLA', 'ECU_TIME_AT_TEMP', 'IGNITORA_LIFE_USAGE',
    'IGNITORB_LIFE_USAGE', 'ECU_STATISTICS', 'ENGINE_STATISTICS', 'USAGE_STATISTICS',
    'LANDING_STATISTICS', 'MC_ATTRIBUTES', 'MC_HISTORY_ATTRIBUTES'
}

EVENT_SECTIONS = {
    'STARTS', 'TKO_PERFORMANCE', 'CLM_PERFORMANCE', 'CRU_PERFORMANCE', 'ROLLDOWN',
    'INCIDENT_RECORD_A', 'INCIDENT_RECORD_B', 'ENGINE_REDLINE', 'IGNITION_RECORD', 'CHIP',
    'FF_ATTRIBUTES', 'ENGINE_TIME_AT_PLA', 'ECU_TIME_AT_TEMP', 'IGNITORA_LIFE_USAGE',
    'IGNITORB_LIFE_USAGE'
}

IDENTITY_SECTIONS = {'ECU_STATISTICS', 'ENGINE_STATISTICS'}

def split_txt_sections_from_lines(lines):
    sections = {}
    current = None
    buffer = []
    for line in lines:
        s = line.strip()
        if s in TXT_SECTION_NAMES:
            if current is not None:
                sections[current] = buffer
            current = s
            buffer = []
        else:
            if current is not None:
                buffer.append(line)
    if current is not None:
        sections[current] = buffer
    return sections

def section_has_tabular_rows(lines):
    header_seen = False
    data_rows = 0
    header_cols = 0
    for line in lines:
        s = line.strip()
        if not s or set(s) == {'='} or s.startswith('<br') or s.startswith('&lt;br'):
            continue
        if '\t' not in line:
            continue
        parts = [p.strip() for p in line.rstrip('\n').split('\t')]
        non_empty = [p for p in parts if p != '']
        if not header_seen:
            header_seen = True
            header_cols = len(non_empty)
            continue
        if len(non_empty) >= 2:
            data_rows += 1
    return header_seen, data_rows, header_cols

def scan_txt_file_v2(path):
    try:
        text = path.read_text(encoding='utf-8', errors='ignore')
        lines = text.splitlines(True)
        sections = split_txt_sections_from_lines(lines)

        section_rows = {}
        section_header_cols = {}
        event_sections_with_rows = []
        identity_sections_with_rows = []
        total_event_rows = 0
        total_tabular_rows = 0

        for section, section_lines in sections.items():
            has_header, data_rows, header_cols = section_has_tabular_rows(section_lines)
            section_rows[section] = data_rows
            section_header_cols[section] = header_cols
            total_tabular_rows += data_rows
            if section in EVENT_SECTIONS and data_rows > 0:
                event_sections_with_rows.append(section)
                total_event_rows += data_rows
            if section in IDENTITY_SECTIONS and data_rows > 0:
                identity_sections_with_rows.append(section)

        has_engine_identity_text = (
            'ENGINE SERIAL' in text.upper()
            or 'ENGINE_SERIAL' in text.upper()
            or 'ENGINE S/N' in text.upper()
            or 'ESN' in text.upper()
        )

        has_ecu_identity_text = (
            'ECU SERIAL' in text.upper()
            or 'ECU_SERIAL' in text.upper()
            or 'ECU S/N' in text.upper()
        )

        has_time_text = (
            'GMT DATE' in text.upper()
            or 'GMT TIME' in text.upper()
            or 'ECU OPERATING TIME' in text.upper()
            or 'OPERATING TIME' in text.upper()
        )

        usable_for_event_merge = (
            len(event_sections_with_rows) > 0
            and total_event_rows > 0
        )

        metadata_only = (
            not usable_for_event_merge
            and len(identity_sections_with_rows) > 0
        )

        empty_or_unparsed = (
            len(sections) == 0
            or total_tabular_rows == 0
        )

        return {
            'txt_file': str(path),
            'filename': path.name,
            'size_bytes': path.stat().st_size,
            'sections': '|'.join(sorted(sections.keys())),
            'section_count': len(sections),
            'event_sections_with_rows': '|'.join(sorted(event_sections_with_rows)),
            'event_section_count': len(event_sections_with_rows),
            'identity_sections_with_rows': '|'.join(sorted(identity_sections_with_rows)),
            'identity_section_count': len(identity_sections_with_rows),
            'total_event_rows_estimate': total_event_rows,
            'total_tabular_rows_estimate': total_tabular_rows,
            'has_engine_identity_text': has_engine_identity_text,
            'has_ecu_identity_text': has_ecu_identity_text,
            'has_time_text': has_time_text,
            'usable_for_event_merge': usable_for_event_merge,
            'metadata_only': metadata_only,
            'empty_or_unparsed': empty_or_unparsed,
            'section_row_summary': '|'.join([f'{k}:{v}' for k, v in sorted(section_rows.items())]),
        }
    except Exception as ex:
        return {
            'txt_file': str(path),
            'filename': path.name,
            'size_bytes': None,
            'sections': '',
            'section_count': 0,
            'event_sections_with_rows': '',
            'event_section_count': 0,
            'identity_sections_with_rows': '',
            'identity_section_count': 0,
            'total_event_rows_estimate': 0,
            'total_tabular_rows_estimate': 0,
            'has_engine_identity_text': False,
            'has_ecu_identity_text': False,
            'has_time_text': False,
            'usable_for_event_merge': False,
            'metadata_only': False,
            'empty_or_unparsed': True,
            'error': str(ex),
        }

txt_files = sorted(TXT_ROOT.rglob('*.txt'))
print(f'TXT files found: {len(txt_files):,}')

scan_rows = []
print(f'Concurrent scan workers: {MAX_WORKERS:,}')
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(scan_txt_file_v2, txt): txt for txt in txt_files}
    for idx, future in enumerate(as_completed(futures), start=1):
        scan_rows.append(future.result())
        if idx == 1 or idx % 5000 == 0 or idx == len(txt_files):
            print(f'Scanned {idx:,}/{len(txt_files):,}: {futures[future].name}')

scan_df = pd.DataFrame(scan_rows)
usable_df = scan_df[scan_df['usable_for_event_merge'] == True].copy()
metadata_df = scan_df[(scan_df['usable_for_event_merge'] != True) & (scan_df['metadata_only'] == True)].copy()
unparsed_df = scan_df[scan_df['empty_or_unparsed'] == True].copy()

scan_df.to_csv(TXT_SCAN_REPORT, index=False)
usable_df.to_csv(USABLE_LIST_FILE, index=False)
metadata_df.to_csv(METADATA_ONLY_LIST_FILE, index=False)
unparsed_df.to_csv(LOG_DIR / 'unparsed_or_empty_txt_files.csv', index=False)

banner('TXT CONTENT SCAN SUMMARY V2')
print(f'TXT scanned              : {len(scan_df):,}')
print(f'Usable event TXT         : {len(usable_df):,}')
print(f'Metadata-only TXT        : {len(metadata_df):,}')
print(f'Unparsed or empty TXT    : {len(unparsed_df):,}')
print(f'Total event rows estimate: {int(scan_df["total_event_rows_estimate"].sum()):,}')
print(f'Total tabular row estimate: {int(scan_df["total_tabular_rows_estimate"].sum()):,}')
print('Scan report              :', TXT_SCAN_REPORT)
print('Usable list              :', USABLE_LIST_FILE)
print('Metadata-only list       :', METADATA_ONLY_LIST_FILE)
print('Unparsed/empty list      :', LOG_DIR / 'unparsed_or_empty_txt_files.csv')

print('\nTop event section combinations:')
print(
    scan_df['event_sections_with_rows']
    .value_counts(dropna=False)
    .head(20)
)

display(scan_df.sort_values('total_event_rows_estimate', ascending=False).head(20))


In [ ]:
from pathlib import Path

import pandas as pd


REPORT_PATH = Path(
    "/raid3/e296408/All_ECFRs_working/logs/"
    "raw_to_txt_conversion_report_20260919_113606.csv"
)


report = pd.read_csv(
    REPORT_PATH
)


failed_other = report.loc[
    report["status"].eq(
        "FAILED_OTHER"
    )
].copy()


def final_traceback_line(value):

    if pd.isna(value):
        return "NO_STDERR"

    lines = [
        line.strip()
        for line in str(value).splitlines()
        if line.strip()
    ]

    if not lines:
        return "EMPTY_STDERR"

    return lines[-1]


failed_other[
    "final_error_line"
] = failed_other[
    "stderr"
].map(
    final_traceback_line
)


error_summary = (
    failed_other.groupby(
        "final_error_line",
        dropna=False,
    )
    .agg(
        files=(
            "raw_key",
            "size",
        ),

        unique_raw_hashes=(
            "raw_sha256",
            "nunique",
        ),

        minimum_raw_bytes=(
            "raw_size_bytes",
            "min",
        ),

        median_raw_bytes=(
            "raw_size_bytes",
            "median",
        ),

        maximum_raw_bytes=(
            "raw_size_bytes",
            "max",
        ),

        example_raw_file=(
            "raw_file",
            "first",
        ),
    )
    .reset_index()
    .sort_values(
        "files",
        ascending=False,
    )
)


pd.set_option(
    "display.max_colwidth",
    500,
)


display(
    error_summary.head(50)
)


error_summary.to_csv(
    REPORT_PATH.parent
    / (
        "raw_to_txt_failed_other_"
        "final_error_summary_"
        "20260919_113606.csv"
    ),
    index=False,
)